In [ ]:
import json
import csv
import struct

input_file = "Normal_air_10B.bmerawdata"
output_file = "Normal_air_10B.csv"

def convert_json_bme(input_file, output_file):
    """Convert JSON-based .bmerawdata to CSV"""
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Detect key that contains sensor samples
    samples = data.get("samples") or data.get("data") or data

    if not isinstance(samples, list):
        raise ValueError("No valid list of samples found in JSON file")

    # Use the keys of the first entry as CSV headers
    headers = samples[0].keys()

    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(samples)

    print(f"✅ JSON data successfully converted to {output_file}")

def convert_binary_bme(input_file, output_file):
    """Convert binary-based .bmerawdata (rough example) to CSV"""
    # The exact binary layout depends on your BME688 firmware/export settings.
    # Below is a generic placeholder to show structure.
    with open(input_file, "rb") as f:
        raw = f.read()

    # Example structure: [timestamp, temperature, humidity, pressure, gas]
    # Each value could be a 4-byte float or int (depends on your data)
    record_size = 20  # 5 values * 4 bytes each
    records = []

    for i in range(0, len(raw), record_size):
        chunk = raw[i:i+record_size]
        if len(chunk) < record_size:
            break
        timestamp, temp, hum, pres, gas = struct.unpack("<fffff", chunk)
        records.append({
            "timestamp": timestamp,
            "temperature": temp,
            "humidity": hum,
            "pressure": pres,
            "gas_resistance": gas
        })

    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=records[0].keys())
        writer.writeheader()
        writer.writerows(records)

    print(f"✅ Binary data successfully converted to {output_file}")

# --- Try reading as text (JSON) first ---
try:
    with open(input_file, "r", encoding="utf-8") as f:
        first_chars = f.read(100)
        if first_chars.strip().startswith("{"):
            convert_json_bme(input_file, output_file)
        else:
            raise ValueError
except Exception:
    print("ℹ️ File not JSON — trying binary mode...")
    convert_binary_bme(input_file, output_file)
